In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

def calculate_true_probas(h, d, a):
    """Calcule les probabilités sans la marge du bookmaker (vectorisé)"""
    inv = 1.0 / np.array([h, d, a])
    return inv / np.sum(inv, axis=0)

# 1. Chargement du fichier de Premier League
sys.path.append(str(Path.cwd().parent))
data_path_csv = Path.cwd().parent / "data" / "Scraped" / "England_2023-2024.csv"
df = pd.read_csv(data_path_csv)

# 2. On s'assure que les colonnes d'ouverture (Avg) et de fermeture (AvgC) existent
df = df.dropna(subset=['AvgH', 'AvgD', 'AvgA', 'AvgCH', 'AvgCD', 'AvgCA'])

# Le fichier contient 380 matchs (38 journées de 10 matchs) dans l'ordre chronologique.
# On ajoute une colonne "Journée" (Matchday) approximative
df['Matchday'] = (df.index // 10) + 1

# 3. Calcul des probabilités d'Ouverture et de Fermeture pour l'équipe à Domicile (Home)
probas_open = calculate_true_probas(df['AvgH'], df['AvgD'], df['AvgA'])
probas_close = calculate_true_probas(df['AvgCH'], df['AvgCD'], df['AvgCA'])

# 4. Calcul du Drift (Fermeture - Ouverture)
df['Drift_Home'] = probas_close[0] - probas_open[0]

# 5. Séparation en deux univers : La "Saison Régulière" (J1/J2) vs "La Fin de Saison" (J3)
mask_saison = df['Matchday'] <= 30
mask_chaos = df['Matchday'] >= 35

std_saison = df[mask_saison]['Drift_Home'].std()
std_chaos = df[mask_chaos]['Drift_Home'].std()

print("--- CALIBRATION EMPIRIQUE DU DRIFT ---")
print(f"Écart-type (σ) Saison Régulière (équivalent Poule J1/J2) : {std_saison:.4f}")
print(f"Écart-type (σ) Fin de Saison (équivalent Poule J3)       : {std_chaos:.4f}")

# Petit bonus : Regarder le match avec le plus gros drift de l'année
max_drift_idx = df['Drift_Home'].abs().idxmax()
print(f"\nMatch avec le plus gros drift : {df.loc[max_drift_idx, 'HomeTeam']} vs {df.loc[max_drift_idx, 'AwayTeam']} (Journée {df.loc[max_drift_idx, 'Matchday']})")
print(f"-> Proba Victoire {df.loc[max_drift_idx, 'HomeTeam']} (Ouverture) : {probas_open[0][max_drift_idx]*100:.1f}%")
print(f"-> Proba Victoire {df.loc[max_drift_idx, 'HomeTeam']} (Fermeture) : {probas_close[0][max_drift_idx]*100:.1f}%")

--- CALIBRATION EMPIRIQUE DU DRIFT ---
Écart-type (σ) Saison Régulière (équivalent Poule J1/J2) : 0.0230
Écart-type (σ) Fin de Saison (équivalent Poule J3)       : 0.0334

Match avec le plus gros drift : Brighton vs Man United (Journée 38)
-> Proba Victoire Brighton (Ouverture) : 44.2%
-> Proba Victoire Brighton (Fermeture) : 36.7%
